In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/rahelyabebal/train-data/train.npy
/kaggle/input/datasets/rahelyabebal/train-data/vocab.txt
/kaggle/input/notebooks/naomimeseret/amharic-n-gram-model-building-and-training-membe/__notebook__.ipynb
/kaggle/input/notebooks/naomimeseret/amharic-n-gram-model-building-and-training-membe/__output__.json
/kaggle/input/notebooks/naomimeseret/amharic-n-gram-model-building-and-training-membe/__results__.html
/kaggle/input/notebooks/naomimeseret/amharic-n-gram-model-building-and-training-membe/custom.css
/kaggle/input/notebooks/naomimeseret/amharic-n-gram-model-building-and-training-membe/.virtual_documents/__notebook_source__.ipynb
/kaggle/input/notebooks/naomimeseret/amharic-n-gram-model-building-and-training-membe/code/src/ngram_disk.py
/kaggle/input/notebooks/naomimeseret/amharic-n-gram-model-building-and-training-membe/code/src/ngram_model.py
/kaggle/input/notebooks/naomimeseret/amharic-n-gram-model-building-and-training-membe/code/src/preprocess_lib.py
/kaggle/input/no

In [12]:
import sys
from pathlib import Path
import shutil

code_dir = Path("/kaggle/working/code")

# If the folder already exists, remove it so we can do a fresh clone
if code_dir.exists():
    shutil.rmtree(code_dir)

# Clone your project repository into Kaggle's working directory
!git clone https://github.com/NaomiMeseret/amharic-ngram-language-model /kaggle/working/code

# Add the cloned repository to Python's search path
sys.path.insert(0, "/kaggle/working/code")

print("Repository cloned and path configured successfully!")

Cloning into '/kaggle/working/code'...
remote: Enumerating objects: 294, done.
remote: Counting objects: 100% (294/294), done.
remote: Compressing objects: 100% (202/202), done.
remote: Total 294 (delta 109), reused 218 (delta 84), pack-reused 0 (from 0)
Receiving objects: 100% (294/294), 4.08 MiB | 22.72 MiB/s, done.
Resolving deltas: 100% (109/109), done.
Repository cloned and path configured successfully!


In [8]:
import zipfile
from pathlib import Path
from src.ngram_model import NGramModel

# Locate and extract the trained model zip from Kaggle inputs
input_dir = Path("/kaggle/input")
zip_path = next(input_dir.rglob("trained_model.zip"), None)

if zip_path and zip_path.is_file():
    print(f"Found trained model archive at: {zip_path}")
    target_dir = Path("/kaggle/working/ngram_results")
    target_dir.mkdir(parents=True, exist_ok=True)
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(target_dir)
    print("Model extracted successfully!")
else:
    print("Model archive not found. Make sure Naomi's notebook output is added as input.")

# Load the model
model_dir = Path("/kaggle/working/ngram_results/model")
model = NGramModel.load(model_dir, mmap_mode="r")
print(f"Successfully loaded model with max order: {model.max_order}")

Found trained model archive at: /kaggle/input/notebooks/naomimeseret/amharic-n-gram-model-building-and-training-membe/ngram_results/final_downloads/trained_model.zip
Model extracted successfully!
Successfully loaded model with max order: 4


In [20]:
from pathlib import Path
import numpy as np

# Search the entire Kaggle directory for test.npy
all_tests = list(Path("/kaggle").rglob("test.npy"))

if all_tests:
    test_path = all_tests[0]
    print(f"Found test.npy at: {test_path}")
    
    test_ids = np.load(test_path, mmap_mode="r", allow_pickle=False)
    print(f"Successfully loaded test array with shape: {test_ids.shape}")
else:
    print("test.npy not found yet. Check the left-hand panel to ensure the upload progress ring has finished spinning!")

Found test.npy at: /kaggle/input/datasets/rahelyabebal/test-data/test.npy
Successfully loaded test array with shape: (8579227,)


### Task 3: Extrinsic Evaluation 

We evaluate model utility on two core downstream tasks:

Next-Word Prediction (Autocomplete): Measures how accurately the model predicts the upcoming token given a preceding context window, reported via Top-1, Top-3, and Top-5 accuracy.

Real-vs-Shuffled Sentence Discrimination (Fluency): Evaluates whether the model assigns higher aggregate log-probabilities to natural Amharic sentences versus scrambled word-order equivalents, serving as a baseline for autocorrect and fluency scoring.

In [26]:
import random, time
import numpy as np
import pandas as pd
from src.ngram_model import iter_encoded_sentences
import src.extrinsic_eval as ext

# 1. Decode the flat ID stream into list-of-int sentences (validates BOS/EOS pairing)
all_sentences = [s for s in iter_encoded_sentences(test_ids) if len(s) >= 5]
print(f"{len(all_sentences):,} usable test sentences in total")

# 2. Random samples (fixed seed so results are reproducible)
rnd = random.Random(42)
nw_sample   = rnd.sample(all_sentences, min(2000, len(all_sentences)))
disc_sample = rnd.sample(all_sentences, min(1000, len(all_sentences)))  # backoff scoring is slower

# 3. Next-word prediction (exact-order, no backoff)
nw_results = []
for n in range(1, 5):
    t0 = time.time()
    res = ext.next_word_eval(model, nw_sample, order=n, ks=(1, 3, 5))
    res["order"] = n
    nw_results.append(res)
    print(f"next-word order {n} done in {time.time() - t0:.0f}s")

df_nw = pd.DataFrame(nw_results)
print("\n--- Next-Word Prediction Benchmark ---")
print(df_nw[["order", "top1", "top3", "top5", "coverage", "evaluated", "skipped_unk_eos"]]
      .to_string(index=False))

# 4. Real vs. shuffled discrimination (backoff-smoothed log-probabilities)
disc_results = []
for n in range(1, 5):
    t0 = time.time()
    res = ext.discrimination_eval(model, disc_sample, order=n, method="backoff")
    res["order"] = n
    disc_results.append(res)
    print(f"discrimination order {n} done in {time.time() - t0:.0f}s")

df_disc = pd.DataFrame(disc_results)
print("\n--- Real vs. Shuffled Sentence Discrimination ---")
print(df_disc[["order", "accuracy", "ties", "n"]].to_string(index=False))

489,374 usable test sentences in total
next-word order 1 done in 48s
next-word order 2 done in 440s
next-word order 3 done in 156s
next-word order 4 done in 119s

--- Next-Word Prediction Benchmark ---
 order     top1     top3     top5  coverage  evaluated  skipped_unk_eos
     1 0.017499 0.044473 0.054636  1.000000      26173             6189
     2 0.108012 0.181905 0.220838  1.000000      26173             6189
     3 0.131662 0.196577 0.225041  0.854621      26173             6189
     4 0.114240 0.149849 0.163031  0.541512      26173             6189
discrimination order 1 done in 1s
discrimination order 2 done in 156s
discrimination order 3 done in 261s
discrimination order 4 done in 317s

--- Real vs. Shuffled Sentence Discrimination ---
 order  accuracy  ties   n
     1  0.000000   998 998
     2  0.988978     0 998
     3  0.986974     0 998
     4  0.988978     0 998


In [27]:
import math, random
from src.ngram_model import BOS, EOS, UNK, normalize_and_tokenize

# ---------- edit these ----------
CUSTOM = [
    "ኢትዮጵያ በአፍሪካ ቀንድ ትገኛለች።",
    "አዲስ አበባ የኢትዮጵያ ዋና ከተማ ናት።",
]                      # replace with your own sentences
N_FROM_TEST = 2        # extra random real sentences from the test set
ORDERS = (2, 3, 4)
# --------------------------------

V, W2I = model.vocabulary, model.word_to_id
SPECIAL = {0, 1, 2, 3}

def logp(ids, order):
    """log P(sentence) with backoff, </s> included, same framing as training."""
    framed = [BOS] * (order - 1) + ids + [EOS]
    return sum(model.logprob(framed[i], framed[i - order + 1:i], method="backoff")
               for i in range(order - 1, len(framed)))

def exact_suggestions(prefix_ids, order, k=5):
    ctx = ([BOS] * (order - 1) + prefix_ids)[-(order - 1):]
    pairs = [(t, c) for t, c in model.orders[order].next_counts(ctx) if t not in SPECIAL]
    pairs.sort(key=lambda p: -p[1])
    return [V[t] for t, _ in pairs[:k]]

# collect sentences as word lists
items = [normalize_and_tokenize(t) for t in CUSTOM]
if N_FROM_TEST and "all_sentences" in globals():
    rnd = random.Random(5)
    pool = [s for s in all_sentences if 8 <= len(s) <= 12 and UNK not in s]
    items += [[V[t] for t in s[1:-1]] for s in rnd.sample(pool, N_FROM_TEST)]

rng = random.Random(1)
for words in items:
    ids = [W2I.get(w, UNK) for w in words]
    shuffled = ids[:]
    for _ in range(20):
        rng.shuffle(shuffled)
        if shuffled != ids: break

    print("=" * 70)
    print("REAL    :", " ".join(words))
    print("SHUFFLED:", " ".join(V[i] for i in shuffled))

    print("\nTest 1: which is more likely? (higher log-prob wins)")
    for n in ORDERS:
        a, b = logp(ids, n), logp(shuffled, n)
        print(f"  n={n}: real {a:8.1f} | shuffled {b:8.1f} -> {'real wins ✓' if a > b else 'WRONG ✗'}")

    print("\nTest 2: next-word suggestions (exact n-gram, no backoff)")
    for i in range(1, len(ids)):
        prefix = ids[:i]
        line = [f"n={n} " + ("✓" if words[i] in (s := exact_suggestions(prefix, n)) else "✗") for n in ORDERS]
        print(f"  {' '.join(words[:i]):<35} -> actual: {words[i]:<12} {'  '.join(line)}")
        print(f"      n=3 suggests: {', '.join(exact_suggestions(prefix, 3)) or '(context unseen)'}")

REAL    : ኢትዮጵያ በአፍሪካ ቀንድ ትገኛለች
SHUFFLED: ትገኛለች ኢትዮጵያ ቀንድ በአፍሪካ

Test 1: which is more likely? (higher log-prob wins)
  n=2: real    -25.9 | shuffled    -55.8 -> real wins ✓
  n=3: real    -25.1 | shuffled    -57.2 -> real wins ✓
  n=4: real    -25.6 | shuffled    -57.2 -> real wins ✓

Test 2: next-word suggestions (exact n-gram, no backoff)
  ኢትዮጵያ                               -> actual: በአፍሪካ        n=2 ✗  n=3 ✗  n=4 ✗
      n=3 suggests: ውስጥ, ነገ, በክብር, ለዘለአለም, በአለም
  ኢትዮጵያ በአፍሪካ                         -> actual: ቀንድ          n=2 ✓  n=3 ✓  n=4 ✓
      n=3 suggests: ቀንድ, ውስጥ, ደረጃ, አህጉር, ብሎም
  ኢትዮጵያ በአፍሪካ ቀንድ                     -> actual: ትገኛለች        n=2 ✗  n=3 ✗  n=4 ✗
      n=3 suggests: አካባቢ, ላይ, ውስጥ, ሰላምና, ሰላም
REAL    : አዲስ አበባ የኢትዮጵያ ዋና ከተማ ናት
SHUFFLED: ከተማ ናት አበባ ዋና አዲስ የኢትዮጵያ

Test 1: which is more likely? (higher log-prob wins)
  n=2: real    -29.0 | shuffled    -58.7 -> real wins ✓
  n=3: real    -19.5 | shuffled    -65.6 -> real wins ✓
  n=4: real    -16.3 | shuffled    -